# 01 — Setup: NorthwindDW no DuckDB

Cria o banco DuckDB, schemas e todas as tabelas (equivalente ao `01_setup.sql` do SQL Server).

**Estrutura:**
```
duckdb/northwind_dw.duckdb
├── bronze   — cópias fiéis dos CSVs exportados do Northwind
├── gold     — modelo dimensional completo (dims SCD1/SCD2 + fatos)
```

**Pré-requisitos:**
```bash
pip install duckdb
```

In [1]:
import duckdb

DB_PATH = "/workspace/pf_northwind/duckdb/northwind_dw.duckdb"

conn = duckdb.connect(DB_PATH)
print(f"DuckDB {duckdb.__version__} | Banco: {DB_PATH}")

DuckDB 0.10.0 | Banco: /workspace/pf_northwind/duckdb/northwind_dw.duckdb


In [2]:
# ============================================================
# Criar schemas
# ============================================================
for schema in ["bronze", "gold"]:
    conn.execute(f"CREATE SCHEMA IF NOT EXISTS {schema}")
    print(f"Schema '{schema}' OK")

Schema 'bronze' OK
Schema 'gold' OK


In [3]:
# ============================================================
# Bronze — cópias fiéis das fontes + _LoadTimestamp
# ============================================================
bronze_ddl = [
    """
    CREATE TABLE IF NOT EXISTS bronze.customers (
        CustomerID VARCHAR, CompanyName VARCHAR, ContactName VARCHAR,
        ContactTitle VARCHAR, Address VARCHAR, City VARCHAR, Region VARCHAR,
        PostalCode VARCHAR, Country VARCHAR, Phone VARCHAR, Fax VARCHAR,
        _LoadTimestamp TIMESTAMP
    )""",
    """
    CREATE TABLE IF NOT EXISTS bronze.employees (
        EmployeeID INTEGER, LastName VARCHAR, FirstName VARCHAR, Title VARCHAR,
        TitleOfCourtesy VARCHAR, BirthDate TIMESTAMP, HireDate TIMESTAMP,
        Address VARCHAR, City VARCHAR, Region VARCHAR, PostalCode VARCHAR,
        Country VARCHAR, HomePhone VARCHAR, Extension VARCHAR,
        ReportsTo INTEGER, PhotoPath VARCHAR, _LoadTimestamp TIMESTAMP
    )""",
    """
    CREATE TABLE IF NOT EXISTS bronze.products (
        ProductID INTEGER, ProductName VARCHAR, SupplierID INTEGER, CategoryID INTEGER,
        QuantityPerUnit VARCHAR, UnitPrice DOUBLE, UnitsInStock INTEGER,
        UnitsOnOrder INTEGER, ReorderLevel INTEGER, Discontinued BOOLEAN,
        _LoadTimestamp TIMESTAMP
    )""",
    """
    CREATE TABLE IF NOT EXISTS bronze.categories (
        CategoryID INTEGER, CategoryName VARCHAR, Description VARCHAR,
        _LoadTimestamp TIMESTAMP
    )""",
    """
    CREATE TABLE IF NOT EXISTS bronze.suppliers (
        SupplierID INTEGER, CompanyName VARCHAR, ContactName VARCHAR,
        ContactTitle VARCHAR, Address VARCHAR, City VARCHAR, Region VARCHAR,
        PostalCode VARCHAR, Country VARCHAR, Phone VARCHAR, Fax VARCHAR,
        _LoadTimestamp TIMESTAMP
    )""",
    """
    CREATE TABLE IF NOT EXISTS bronze.shippers (
        ShipperID INTEGER, CompanyName VARCHAR, Phone VARCHAR,
        _LoadTimestamp TIMESTAMP
    )""",
    """
    CREATE TABLE IF NOT EXISTS bronze.orders (
        OrderID INTEGER, CustomerID VARCHAR, EmployeeID INTEGER,
        OrderDate TIMESTAMP, RequiredDate TIMESTAMP, ShippedDate TIMESTAMP,
        ShipVia INTEGER, Freight DOUBLE, ShipName VARCHAR, ShipAddress VARCHAR,
        ShipCity VARCHAR, ShipRegion VARCHAR, ShipPostalCode VARCHAR,
        ShipCountry VARCHAR, _LoadTimestamp TIMESTAMP
    )""",
    """
    CREATE TABLE IF NOT EXISTS bronze.order_details (
        OrderID INTEGER, ProductID INTEGER, UnitPrice DOUBLE,
        Quantity INTEGER, Discount DOUBLE, _LoadTimestamp TIMESTAMP
    )""",
    """
    CREATE TABLE IF NOT EXISTS bronze.territories (
        TerritoryID VARCHAR, TerritoryDescription VARCHAR,
        RegionID INTEGER, _LoadTimestamp TIMESTAMP
    )""",
    """
    CREATE TABLE IF NOT EXISTS bronze.region (
        RegionID INTEGER, RegionDescription VARCHAR, _LoadTimestamp TIMESTAMP
    )""",
    """
    CREATE TABLE IF NOT EXISTS bronze.employee_territories (
        EmployeeID INTEGER, TerritoryID VARCHAR, _LoadTimestamp TIMESTAMP
    )"""
]

for ddl in bronze_ddl:
    conn.execute(ddl)

n = conn.execute("SELECT COUNT(*) FROM information_schema.tables WHERE table_schema = 'bronze'").fetchone()[0]
print(f"Bronze: {n} tabelas criadas.")

Bronze: 11 tabelas criadas.


In [4]:
# ============================================================
# Gold — Dimensões
# DimEmployee mantém UNIQUE em EmployeeSK/EmployeeID para integridade
# SCD1 (Category/Supplier/Shipper/Territory) usam INSERT OR REPLACE —
#   requer UNIQUE na natural key; SK não precisa de UNIQUE
# ============================================================
gold_dims_ddl = [
    """
    CREATE TABLE IF NOT EXISTS gold.DimCustomer (
        CustomerSK INTEGER, CustomerID VARCHAR, CompanyName VARCHAR,
        ContactName VARCHAR, ContactTitle VARCHAR, City VARCHAR, Country VARCHAR,
        ValidFrom DATE, ValidTo DATE, IsCurrent BOOLEAN
    )""",
    """
    CREATE TABLE IF NOT EXISTS gold.DimEmployee (
        EmployeeSK INTEGER UNIQUE, EmployeeID INTEGER UNIQUE, FullName VARCHAR,
        Title VARCHAR, HireDate DATE, City VARCHAR, Country VARCHAR,
        ReportsToID INTEGER, ManagerName VARCHAR, TerritoryList VARCHAR,
        RegionName VARCHAR, LoadTimestamp TIMESTAMP
    )""",
    """
    CREATE TABLE IF NOT EXISTS gold.DimProduct (
        ProductSK INTEGER, ProductID INTEGER, ProductName VARCHAR,
        CategoryName VARCHAR, SupplierCompany VARCHAR, UnitPrice DOUBLE,
        QuantityPerUnit VARCHAR, Discontinued BOOLEAN,
        ValidFrom DATE, ValidTo DATE, IsCurrent BOOLEAN
    )""",
    """
    CREATE TABLE IF NOT EXISTS gold.DimCategory (
        CategorySK INTEGER, CategoryID INTEGER UNIQUE,
        CategoryName VARCHAR, Description VARCHAR, LoadTimestamp TIMESTAMP
    )""",
    """
    CREATE TABLE IF NOT EXISTS gold.DimSupplier (
        SupplierSK INTEGER, SupplierID INTEGER UNIQUE,
        CompanyName VARCHAR, City VARCHAR, Country VARCHAR, LoadTimestamp TIMESTAMP
    )""",
    """
    CREATE TABLE IF NOT EXISTS gold.DimShipper (
        ShipperSK INTEGER, ShipperID INTEGER UNIQUE,
        CompanyName VARCHAR, Phone VARCHAR, LoadTimestamp TIMESTAMP
    )""",
    """
    CREATE TABLE IF NOT EXISTS gold.DimTerritory (
        TerritorySK INTEGER, TerritoryID VARCHAR UNIQUE,
        TerritoryDescription VARCHAR, RegionID INTEGER, RegionName VARCHAR,
        LoadTimestamp TIMESTAMP
    )""",
    """
    CREATE TABLE IF NOT EXISTS gold.DimDate (
        DateKey INTEGER UNIQUE, FullDate DATE, Year INTEGER, Quarter INTEGER,
        Month INTEGER, MonthName VARCHAR, Day INTEGER, DayOfWeek INTEGER,
        DayName VARCHAR, IsWeekend BOOLEAN
    )"""
]

for ddl in gold_dims_ddl:
    conn.execute(ddl)

n = conn.execute("SELECT COUNT(*) FROM information_schema.tables WHERE table_schema IN ('silver','gold')").fetchone()[0]
print(f"Silver: {n} tabelas criadas.")

Silver: 8 tabelas criadas.


In [5]:
# ============================================================
# Gold — Fatos
# ============================================================
gold_ddl = [
    """
    CREATE TABLE IF NOT EXISTS gold.FactSales (
        SalesSK INTEGER UNIQUE, OrderDateKey INTEGER, CustomerSK INTEGER,
        ProductSK INTEGER, EmployeeSK INTEGER, ShipperSK INTEGER,
        OrderID INTEGER, ProductID INTEGER, UnitPrice DOUBLE, Quantity INTEGER,
        Discount DOUBLE, GrossRevenue DOUBLE, NetRevenue DOUBLE,
        LoadTimestamp TIMESTAMP
    )""",
    """
    CREATE TABLE IF NOT EXISTS gold.FactOrderFulfillment (
        FulfillmentSK INTEGER UNIQUE, OrderID INTEGER UNIQUE, CustomerSK INTEGER,
        EmployeeSK INTEGER, ShipperSK INTEGER, OrderDateKey INTEGER,
        RequiredDateKey INTEGER, ShippedDateKey INTEGER, Freight DOUBLE,
        ShipCountry VARCHAR, DaysToShip INTEGER, IsLate BOOLEAN,
        LoadTimestamp TIMESTAMP
    )""",
    """
    CREATE TABLE IF NOT EXISTS gold.FactProductStock (
        StockSK INTEGER, SnapshotDateKey INTEGER, ProductSK INTEGER,
        CategorySK INTEGER, UnitsInStock INTEGER, UnitsOnOrder INTEGER,
        ReorderLevel INTEGER, NeedsReorder BOOLEAN, LoadTimestamp TIMESTAMP
    )"""
]

for ddl in gold_ddl:
    conn.execute(ddl)

n = conn.execute("SELECT COUNT(*) FROM information_schema.tables WHERE table_schema = 'gold'").fetchone()[0]
print(f"Gold: {n} tabelas criadas.")

Gold: 11 tabelas criadas.


In [6]:
# Resumo
print("=" * 50)
print("Setup NorthwindDW — DuckDB")
print("=" * 50)
for schema in ["bronze", "gold"]:
    tables = conn.execute(
        f"SELECT table_name FROM information_schema.tables WHERE table_schema = '{schema}' ORDER BY table_name"
    ).fetchall()
    print(f"\n{schema} ({len(tables)} tabelas):")
    for (t,) in tables:
        print(f"  - {t}")

conn.close()
print(f"\nBanco: {DB_PATH}")

Setup NorthwindDW — DuckDB

bronze (11 tabelas):
  - categories
  - customers
  - employee_territories
  - employees
  - order_details
  - orders
  - products
  - region
  - shippers
  - suppliers
  - territories

gold (11 tabelas):
  - DimCategory
  - DimCustomer
  - DimDate
  - DimEmployee
  - DimProduct
  - DimShipper
  - DimSupplier
  - DimTerritory
  - FactOrderFulfillment
  - FactProductStock
  - FactSales

Banco: /workspace/pf_northwind/duckdb/northwind_dw.duckdb
